In [1]:
# ============================================================
# Find NDBC stations in a lat/lon box, download stdmet data,
# summarize availability for 2024-09-24 to 2024-09-29,
# and save to CSV + NetCDF
#
# Notes:
# - Assumes western longitudes are negative.
# - Uses active station metadata from:
#     https://www.ndbc.noaa.gov/activestations.xml
# - Uses historical standard met files from:
#     https://www.ndbc.noaa.gov/data/historical/stdmet/
# ============================================================

import io
import gzip
import requests
import numpy as np
import pandas as pd
import xarray as xr
import xml.etree.ElementTree as ET
from pathlib import Path

# ============================================================
# USER SETTINGS
# ============================================================
OUT_DIR = Path("F:/crs/proj/2025_NOPP_comparison/helene_waves")
OUT_DIR.mkdir(parents=True, exist_ok=True)

START = "2024-09-24"
END   = "2024-09-29"
YEAR  = 2024

LAT_MIN = 24.35
LAT_MAX = 30.19
LON_MIN = -88.33
LON_MAX = -81.78   # assumed corrected from +81.78

ACTIVE_XML_URL = "https://www.ndbc.noaa.gov/activestations.xml"
STD_MET_URL_TEMPLATE = "https://www.ndbc.noaa.gov/data/historical/stdmet/{station}h{year}.txt.gz"

SUMMARY_CSV = OUT_DIR / "ndbc_station_availability_20240924_20240929.csv"
COMBINED_CSV = OUT_DIR / "ndbc_bulk_box_20240924_20240929.csv"
COMBINED_NC = OUT_DIR / "ndbc_bulk_box_20240924_20240929.nc"

# ============================================================
# STATION METADATA
# ============================================================
def fetch_active_ndbc_stations(xml_url=ACTIVE_XML_URL):
    """
    Read NDBC active station metadata from activestations.xml
    """
    r = requests.get(xml_url, timeout=60)
    r.raise_for_status()

    root = ET.fromstring(r.content)

    rows = []
    for st in root.findall(".//station"):
        rows.append({
            "station": st.attrib.get("id", "").strip(),
            "latitude": pd.to_numeric(st.attrib.get("lat"), errors="coerce"),
            "longitude": pd.to_numeric(st.attrib.get("lon"), errors="coerce"),
            "site_elevation_m": pd.to_numeric(st.attrib.get("elev"), errors="coerce"),
            "station_name": st.attrib.get("name", "").strip(),
            "owner": st.attrib.get("owner", "").strip(),
            "met": st.attrib.get("met", "").strip().lower(),
            "currents": st.attrib.get("currents", "").strip().lower(),
            "waterquality": st.attrib.get("waterquality", "").strip().lower(),
            "dart": st.attrib.get("dart", "").strip().lower(),
        })

    df = pd.DataFrame(rows)
    return df


def subset_stations_bbox(df, lat_min, lat_max, lon_min, lon_max):
    """
    Filter station metadata to a geographic bounding box.
    """
    m = (
        (df["latitude"] >= lat_min) &
        (df["latitude"] <= lat_max) &
        (df["longitude"] >= lon_min) &
        (df["longitude"] <= lon_max)
    )
    return df.loc[m].sort_values(["latitude", "longitude"]).reset_index(drop=True)


# ============================================================
# DOWNLOAD + PARSE NDBC STDMET
# ============================================================
def _read_ndbc_text_table_from_gzip_bytes(content, station):
    """
    Parse one NDBC historical stdmet text file (.txt.gz).

    Handles both:
      - files with '#YY MM DD hh mm ...'
      - files with 'YYYY MM DD hh mm ...'
    """
    with gzip.GzipFile(fileobj=io.BytesIO(content)) as gz:
        text = gz.read().decode("utf-8", errors="replace")

    lines = [ln.rstrip() for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2:
        raise ValueError(f"{station}: file is empty or malformed")

    # Identify header line
    header_line = lines[0].lstrip("#").strip()
    cols = header_line.split()

    # Remove a possible units line if present
    data_start = 1
    if len(lines) > 1:
        second = lines[1].strip()
        second_tokens = second.split()
        # units line usually contains non-numeric tokens like degC, m/s, hPa, sec
        if second_tokens and not second_tokens[0].replace("-", "").isdigit():
            data_start = 2

    data_text = "\n".join(lines[data_start:])
    if not data_text.strip():
        raise ValueError(f"{station}: no data rows found")

    df = pd.read_csv(
        io.StringIO(data_text),
        delim_whitespace=True,
        names=cols,
        na_values=["MM", "999", "999.0", "9999", "9999.0", "99.00", "999.00"],
        low_memory=False,
    )

    # Standardize time
    if {"YY", "MM", "DD", "hh", "mm"}.issubset(df.columns):
        yy = pd.to_numeric(df["YY"], errors="coerce")
        year_vals = np.where(yy < 100, yy + 1900, yy)
        df["time"] = pd.to_datetime(
            {
                "year": year_vals,
                "month": pd.to_numeric(df["MM"], errors="coerce"),
                "day": pd.to_numeric(df["DD"], errors="coerce"),
                "hour": pd.to_numeric(df["hh"], errors="coerce"),
                "minute": pd.to_numeric(df["mm"], errors="coerce"),
            },
            errors="coerce",
            utc=True,
        )
    elif {"YYYY", "MM", "DD", "hh", "mm"}.issubset(df.columns):
        df["time"] = pd.to_datetime(
            {
                "year": pd.to_numeric(df["YYYY"], errors="coerce"),
                "month": pd.to_numeric(df["MM"], errors="coerce"),
                "day": pd.to_numeric(df["DD"], errors="coerce"),
                "hour": pd.to_numeric(df["hh"], errors="coerce"),
                "minute": pd.to_numeric(df["mm"], errors="coerce"),
            },
            errors="coerce",
            utc=True,
        )
    else:
        raise ValueError(f"{station}: could not identify time columns")

    # Numeric conversion for commonly used fields
    for c in ["WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD",
              "PRES", "ATMP", "WTMP", "DEWP", "VIS", "PTDY", "TIDE"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["station"] = station

    return df


def download_ndbc_stdmet_station_year(station, year=2024):
    """
    Download one station-year of NDBC historical stdmet data.
    """
    url = STD_MET_URL_TEMPLATE.format(station=station, year=year)
    r = requests.get(url, timeout=120)

    if r.status_code == 404:
        raise FileNotFoundError(f"{station}: no stdmet file for {year}")

    r.raise_for_status()

    df = _read_ndbc_text_table_from_gzip_bytes(r.content, station=station)
    return df


def subset_date_range_inclusive(df, start, end):
    """
    Inclusive subsetting on UTC timestamps.
    """
    if len(df) == 0:
        return df.copy()

    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)

    t0 = pd.Timestamp(start, tz="UTC")
    t1 = pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    out = out[(out["time"] >= t0) & (out["time"] <= t1)].copy()
    return out.sort_values("time").reset_index(drop=True)


# ============================================================
# RE-USED DOWNLOAD WRAPPER
# ============================================================
def download_ndbc_bulk_stats(buoys, start, end, year=2024):
    """
    Download NDBC stdmet data for every buoy in the list.
    """
    per_buoy = {}

    for buoy in buoys:
        try:
            df = download_ndbc_stdmet_station_year(buoy, year=year)
            df = subset_date_range_inclusive(df, start=start, end=end)
            per_buoy[buoy] = df
            print(f"{buoy}: {len(df)} records")
        except Exception as e:
            print(f"{buoy}: FAILED -> {e}")
            per_buoy[buoy] = pd.DataFrame()

    combined = (
        pd.concat(
            [df for df in per_buoy.values() if len(df) > 0],
            ignore_index=True,
            sort=False,
        )
        if any(len(df) > 0 for df in per_buoy.values())
        else pd.DataFrame()
    )

    if len(combined) > 0:
        preferred_cols = [
            "station", "time",
            "WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD",
            "PRES", "ATMP", "WTMP", "DEWP", "VIS", "PTDY", "TIDE",
        ]
        cols = [c for c in preferred_cols if c in combined.columns] + [c for c in combined.columns if c not in preferred_cols]
        combined = combined[cols].sort_values(["station", "time"]).reset_index(drop=True)

    return combined, per_buoy


# ============================================================
# SAVE TO NETCDF
# ============================================================
def build_ndbc_xarray(df, metadata, buoys):
    """
    Convert combined NDBC dataframe to xarray Dataset with dims:
      time, station

    Saves wave variables plus station metadata.
    """
    if len(df) == 0:
        raise ValueError("Input dataframe is empty")

    df = df.copy()
    metadata = metadata.copy()

    # Convert to timezone-naive datetime64[ns] for xarray/netcdf
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert(None)

    time = np.sort(df["time"].dropna().unique()).astype("datetime64[ns]")
    stations = [b for b in buoys if b in df["station"].unique()]
    ntime = len(time)
    nsta = len(stations)

    meta = (
        metadata.drop_duplicates(subset="station")
        .set_index("station")
        .reindex(stations)
    )

    ds = xr.Dataset(
        coords={
            "time": ("time", time),
            "station": np.arange(nsta, dtype=np.int32),
        }
    )

    ds["station_id"] = ("station", np.asarray(stations, dtype="S8"))

    if "station_name" in meta.columns:
        ds["station_name"] = ("station", meta["station_name"].fillna("").astype("S64"))

    for name, units in [
        ("latitude", "degrees_north"),
        ("longitude", "degrees_east"),
        ("site_elevation_m", "m"),
        ("water_depth_m", "m"),
    ]:
        if name in meta.columns:
            ds[name] = ("station", meta[name].to_numpy(dtype=np.float32))
            ds[name].attrs["units"] = units

    var_map = {
        "WVHT": ("wave_height", "m"),
        "DPD":  ("dominant_period", "s"),
        "APD":  ("average_period", "s"),
        "MWD":  ("mean_wave_direction", "deg"),
    }

    time_index = pd.Index(time)
    station_index = {sta: i for i, sta in enumerate(stations)}

    for src_name, (out_name, units) in var_map.items():
        arr = np.full((ntime, nsta), np.nan, dtype=np.float32)

        if src_name in df.columns:
            for sta, g in df.groupby("station"):
                if sta not in station_index:
                    continue

                j = station_index[sta]
                gg = (
                    g[["time", src_name]]
                    .dropna()
                    .drop_duplicates(subset="time")
                    .sort_values("time")
                )

                if len(gg) == 0:
                    continue

                it = time_index.get_indexer(gg["time"].to_numpy().astype("datetime64[ns]"))
                good = it >= 0
                arr[it[good], j] = gg[src_name].to_numpy(dtype=np.float32)[good]

        ds[out_name] = (("time", "station"), arr)
        ds[out_name].attrs["units"] = units
        ds[out_name].attrs["source_name"] = src_name

    return ds


def save_ndbc_to_netcdf(df, metadata, buoys, out_nc):
    ds = build_ndbc_xarray(df, metadata, buoys)
    try:
        ds.to_netcdf(out_nc, engine="netcdf4")
    except Exception:
        ds.to_netcdf(out_nc)
    return ds


# ============================================================
# SUMMARY TABLE
# ============================================================
def build_availability_table(per_buoy, metadata):
    """
    Build a small station summary for the requested time window.
    """
    meta = metadata.set_index("station")
    rows = []

    for station, df in per_buoy.items():
        row = {
            "station": station,
            "station_name": meta.loc[station, "station_name"] if station in meta.index else "",
            "latitude": meta.loc[station, "latitude"] if station in meta.index else np.nan,
            "longitude": meta.loc[station, "longitude"] if station in meta.index else np.nan,
            "has_any_data": False,
            "n_records": 0,
            "has_wave_height": False,
            "has_dominant_period": False,
            "has_average_period": False,
            "has_mean_wave_direction": False,
            "max_wave_height_m": np.nan,
            "time_min": pd.NaT,
            "time_max": pd.NaT,
        }

        if len(df) > 0:
            row["has_any_data"] = True
            row["n_records"] = len(df)
            row["time_min"] = pd.to_datetime(df["time"], utc=True).min()
            row["time_max"] = pd.to_datetime(df["time"], utc=True).max()

            if "WVHT" in df.columns:
                row["has_wave_height"] = df["WVHT"].notna().any()
                row["max_wave_height_m"] = df["WVHT"].max(skipna=True)

            if "DPD" in df.columns:
                row["has_dominant_period"] = df["DPD"].notna().any()

            if "APD" in df.columns:
                row["has_average_period"] = df["APD"].notna().any()

            if "MWD" in df.columns:
                row["has_mean_wave_direction"] = df["MWD"].notna().any()

        rows.append(row)

    summary = pd.DataFrame(rows).sort_values(
        ["has_any_data", "max_wave_height_m", "station"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    return summary


# ============================================================
# PER-STATION CSV SAVE
# ============================================================
def save_per_buoy_csv(per_buoy, metadata, out_dir):
    """
    Save each buoy with data to an individual CSV, adding metadata columns.
    """
    meta = metadata.set_index("station")

    saved = []
    for station, df in per_buoy.items():
        if len(df) == 0:
            continue

        out = df.copy()
        if station in meta.index:
            out["station_name"] = meta.loc[station, "station_name"]
            out["longitude"] = meta.loc[station, "longitude"]
            out["latitude"] = meta.loc[station, "latitude"]
            out["site_elevation_m"] = meta.loc[station, "site_elevation_m"]

        outfile = out_dir / f"ndbc_{station}.csv"
        out.to_csv(outfile, index=False)
        saved.append(outfile)

    return saved


# ============================================================
# RUN
# ============================================================
# 1) active stations in bounding box
stations_all = fetch_active_ndbc_stations()
stations_box = subset_stations_bbox(
    stations_all,
    lat_min=LAT_MIN,
    lat_max=LAT_MAX,
    lon_min=LON_MIN,
    lon_max=LON_MAX,
)

print(f"Active stations in box: {len(stations_box)}")
print(stations_box[["station", "station_name", "latitude", "longitude", "met"]].to_string(index=False))

# 2) download stdmet for these stations
buoys = stations_box["station"].tolist()
df_ndbc, per_buoy = download_ndbc_bulk_stats(
    buoys=buoys,
    start=START,
    end=END,
    year=YEAR,
)

# 3) build summary table
summary = build_availability_table(per_buoy, stations_box)
summary.to_csv(SUMMARY_CSV, index=False)

print("\nAvailability summary:")
print(summary.to_string(index=False))

# 4) save per-station CSV files
saved_csvs = save_per_buoy_csv(per_buoy, stations_box, OUT_DIR)
print(f"\nSaved {len(saved_csvs)} per-station CSV files")

# 5) save combined CSV and NC for stations that actually have data
if len(df_ndbc) > 0:
    df_ndbc.to_csv(COMBINED_CSV, index=False)
    print(f"Saved combined CSV: {COMBINED_CSV}")

    # metadata can optionally include water_depth_m if you have it from another source;
    # active station XML does not provide water depth.
    ds_ndbc = save_ndbc_to_netcdf(
        df=df_ndbc,
        metadata=stations_box,
        buoys=buoys,
        out_nc=COMBINED_NC,
    )
    print(f"Saved combined NetCDF: {COMBINED_NC}")
    print(ds_ndbc)
else:
    print("\nNo station data were downloaded in the requested window.")

Active stations in box: 44
station                                                      station_name  latitude  longitude met
  42095                                             Satan Shoal, FL (244)    24.407    -81.968   y
  sanf1                                                      Sand Key, FL    24.456    -81.877   y
  kywf1                                            8724580 - Key West, FL    24.556    -81.808   y
  42026               C22 - Loop Current Pressure Point Buoy, 70m Isobath    25.171    -83.475   y
  42097                                            Pulley Ridge, FL (226)    25.704    -83.654   y
  42003                             East GULF - 208 NM West of Naples, FL    25.925    -85.616   n
  42023                                 C13 - WFS South Buoy, 50m Isobath    26.010    -83.086   y
  fmrf1                                          8725520 - Fort Myers, FL    26.647    -81.871   y
  venf1                                                        Venice, FL    27.07

C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_42108\1008810096.py:124: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_42108\1008810096.py:124: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


sanf1: 863 records


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_42108\1008810096.py:124: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


kywf1: 1396 records


...

apcf1: 1396 records
42028: FAILED -> 42028: no stdmet file for 2024
apxf1: FAILED -> apxf1: no stdmet file for 2024
apqf1: FAILED -> apqf1: no stdmet file for 2024
ktnf1: FAILED -> ktnf1: no stdmet file for 2024


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_42108\1008810096.py:124: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


42012: 864 records
42031: FAILED -> 42031: no stdmet file for 2024
42357: FAILED -> 42357: no stdmet file for 2024


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_42108\1008810096.py:124: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


pacf1: 1390 records

Availability summary:
station                                                      station_name  latitude  longitude  has_any_data  n_records  has_wave_height  has_dominant_period  has_average_period  has_mean_wave_direction  max_wave_height_m                  time_min                  time_max
  42097                                            Pulley Ridge, FL (226)    25.704    -83.654          True        288             True                 True                True                     True               9.13 2024-09-24 00:26:00+00:00 2024-09-29 23:56:00+00:00
  42036                             WEST TAMPA  - 112 NM WNW of Tampa, FL    28.500    -84.505          True        862             True                 True                True                     True               7.58 2024-09-24 00:00:00+00:00 2024-09-29 23:50:00+00:00
  42095                                             Satan Shoal, FL (244)    24.407    -81.968          True        285             Tru